# 수행자 보류 프레임에서 원판 중심을 단서로 쓰기

기존 선택에 실패했을 때만 원판 중심을 포함하는 유일한 사람 박스를 사용합니다. 다음 프레임의 이력은 고정합니다.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").exists())
sys.path.insert(0, str(root))
import pandas as pd
import matplotlib.pyplot as plt
from wlcv.evaluation.plate_anchor_performer import run, compare_timing
from wlcv.evaluation.train_knee_pass_coverage import PARENT, extract
sources = sorted(PARENT.glob("*/frames.csv"))
source = sources[-1].parent if sources else extract()
out = root / "outputs/plate_anchor_performer"
out.mkdir(exist_ok=True)
recompute = False

In [ ]:
if recompute or not (out/"frames.csv").exists():
    run(source).to_csv(out/"frames.csv",index=False)
frames = pd.read_csv(out/"frames.csv")
counts = frames.groupby("video")[["original_paired","paired"]].sum()
display(counts)
counts.plot.bar(ylabel="Frames with both coordinates",rot=20)
plt.tight_layout()
plt.show()

## 같은 ready~drive 범위와 시드에서 TCN 비교

수동 통과 시점은 평가 구간을 자르는 데 쓰지 않습니다. 각 열의 결측은 판정 보류입니다.

In [ ]:
runs = sorted(out.glob("folds/*/scores.csv"))
if recompute or not runs:
    fold, scores = compare_timing(source,out)
else:
    scores = pd.read_csv(runs[-1])
display(scores[scores.seed.fillna(0).eq(0)].pivot(index="video",columns="method",values="outside_ms").round(1))